# Segmentation

The [Quickstart](quickstart.ipynb) shrank the data by keeping fewer **periods** — a
year of days became a handful of typical days. **Segmentation** is a second,
independent lever that works *inside* each period.

A typical day still has 24 hourly values. Many of those hours are nearly
identical — overnight load barely moves for hours. Segmentation merges runs of
similar consecutive steps into a single **segment** with one value, so a 24-hour
day might be described by just 6 segments. Segments are **variable length**: long
where the profile is flat, short where it changes quickly.

It is easiest to just see it.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import SegmentConfig

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## See it: 24 hours → 6 segments

Aggregate to 6 typical days, but describe each day with **6 segments** instead of
24 hourly steps:

In [ ]:
result = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)

Now compare the **original** hourly load with the **reconstructed** series built
from the segmented typical days. The reconstruction is a step function — each flat
step is one segment, held constant across the hours it covers. Where the original
is busy (the daily ramp), segments are short; where it is calm (overnight), one
long segment spans many hours.

Here is one week — each flat step is a segment. (The plot is interactive, so you can zoom further into a single day.)

In [ ]:
week = slice("2010-01-11", "2010-01-17")
result.plot.compare(columns=["Load"], time_slice=week, color="source")

That is the whole idea: replace 24 hourly values with a few representative steps,
chosen to follow the shape of the data.

## Where the boundaries fall

Because segments are variable length, they absorb more hours where the signal is
steady and fewer where it moves. Averaged across the typical days, the segment
lengths are uneven:

In [ ]:
result.plot.segment_durations()

## Why bother: the budget

Segmentation multiplies with the number of periods. Without it, 6 typical days
cost 6 × 24 = 144 time steps; with 6 segments each, 6 × 6 = 36 — a quarter as many.
The accuracy cost is small and read the same way as always:

In [ ]:
plain = tsam.aggregate(data, n_clusters=6, period_duration="1D")
pd.DataFrame(
    {
        "6 days x 24 h (144 steps)": plain.accuracy.rmse,
        "6 days x 6 seg (36 steps)": result.accuracy.rmse,
    }
).round(4)

So you have two dials: how many typical periods, and how fine each one is.
[Sizing & tuning](sizing_tuning.ipynb) searches both for the best combination at a
target size.